# Create ZonMw Awards (GRANT PATTERN, Netherlands)

Health research and care innovation projects from ZonMw's official Projects site:

- Source search: https://projecten.zonmw.nl/nl/search
- JSON:API source: https://projecten.zonmw.nl/nl/jsonapi/node/project
- S3 location: `s3a://openalex-ingest/awards/zonmw/zonmw_projects.parquet`

**Source decision:** The tracker previously marked ZonMw blocked as web-only, but the official Projects site exposes Drupal JSON:API records under `/nl/jsonapi/node/project`. This is preferred over scraping the rendered search UI. The API exposes project number, title, start/end dates, summaries/results, project leader, responsible organization taxonomy terms, related programs/subsidy relationships, subjects, and canonical paths.

**Funder:** `F4320321007` — ZonMw. DOI `10.13039/501100001826`.

**Expected local scrape:** The search page advertised 17,991 Project records on 2026-05-21. The script filters the one template placeholder row with no real project number. It preserves the native ZonMw `field_project_number` as `project_number`; the shipped `funder_award_id` is `zonmw-{project_number}-{drupal_internal_nid}` because the source contains a small number of real duplicate project numbers for distinct project pages. Duplicate shipped slugs raise in the downloader.

**Amount/currency waiver:** The public JSON:API includes `field_project_budget` and `field_project_budget_api`, but local API samples show these project-budget fields are null. This notebook maps amount/currency only when ZonMw publishes `field_project_budget_api`; otherwise amount/currency remain NULL by source authority. Do not infer amounts from program-level budgets. One source row currently has an impossible start date (`1024-04-22`); the transform suppresses impossible dates/years and falls back to the end year for `start_year` when needed so the row is not silently lost.

**Contractor handoff:** Script/notebook are prepared locally. Contractor has no S3/Databricks access; admin must upload parquet, run this notebook, run verification, and only then wire scheduled jobs if approved.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.zonmw_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/zonmw/zonmw_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_raw_rows FROM openalex.awards.zonmw_raw;


## Step 1.5: Inspect raw + money-field scan

Mandatory grant-pattern inspection. `field_project_budget` / `project_budget_api` are preserved from the official source, but they are expected to be mostly or entirely NULL in the public API. Confirm no other money-shaped fields exist before accepting the waiver.


In [ ]:
%sql
DESCRIBE openalex.awards.zonmw_raw;


In [ ]:
%sql
SELECT
    project_number,
    display_name,
    project_leader_name,
    primary_organization,
    project_budget_text,
    project_budget_api,
    currency,
    start_date,
    end_date,
    related_programs_json,
    related_subsidies_json,
    main_subject,
    landing_page_url
FROM openalex.awards.zonmw_raw
LIMIT 10;


In [ ]:
%sql
-- Money-field scan per runbook section 1.5
SELECT column_name FROM (DESCRIBE openalex.awards.zonmw_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|subsid';


In [ ]:
%sql
SELECT
    MIN(TRY_CAST(project_budget_api AS DOUBLE)) AS min_project_budget,
    MAX(TRY_CAST(project_budget_api AS DOUBLE)) AS max_project_budget,
    ROUND(AVG(TRY_CAST(project_budget_api AS DOUBLE)), 2) AS avg_project_budget,
    COUNT(project_budget_api) AS non_null_project_budget,
    COUNT(project_budget_text) AS non_null_project_budget_text,
    COUNT(*) AS total_rows,
    collect_set(currency) AS currencies
FROM openalex.awards.zonmw_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(project_number) AS has_project_number,
    COUNT(project_leader_name) AS has_project_leader,
    COUNT(primary_organization) AS has_primary_organization,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(related_programs_json) AS has_related_programs,
    COUNT(main_subject) AS has_main_subject,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT slug) AS distinct_slugs
FROM openalex.awards.zonmw_raw;


## Step 1.6: Fail-fast - verify ZonMw funder exists

The Step 2 transform joins to `openalex.common.funder`. This must return exactly one row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321007;


## Step 2: Transform to OpenAlex awards schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.zonmw_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321007
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(project_budget_api AS DOUBLE) AS amount_double,
        CASE
            WHEN YEAR(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) BETWEEN 1900 AND 2035
                THEN TRY_TO_DATE(start_date, 'yyyy-MM-dd')
            ELSE NULL
        END AS start_date_cast,
        CASE
            WHEN YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) BETWEEN 1900 AND 2035
                THEN TRY_TO_DATE(end_date, 'yyyy-MM-dd')
            ELSE NULL
        END AS end_date_cast,
        COALESCE(
            CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1900 AND 2035 THEN TRY_CAST(start_year AS INT) END,
            CASE WHEN YEAR(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) BETWEEN 1900 AND 2035 THEN YEAR(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) END,
            CASE WHEN YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) BETWEEN 1900 AND 2035 THEN YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) END
        ) AS start_year_int,
        COALESCE(
            CASE WHEN TRY_CAST(end_year AS INT) BETWEEN 1900 AND 2035 THEN TRY_CAST(end_year AS INT) END,
            CASE WHEN YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) BETWEEN 1900 AND 2035 THEN YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) END
        ) AS end_year_int,
        NULLIF(TRIM(project_leader_name), '') AS leader_full_name,
        NULLIF(TRIM(primary_organization), '') AS leader_affiliation_name,
        CONCAT_WS('\\n\\n',
            NULLIF(TRIM(description), ''),
            CASE WHEN NULLIF(TRIM(application_summary), '') IS NOT NULL THEN CONCAT('Application summary: ', TRIM(application_summary)) END,
            CASE WHEN NULLIF(TRIM(progress_summary), '') IS NOT NULL THEN CONCAT('Progress summary: ', TRIM(progress_summary)) END,
            CASE WHEN NULLIF(TRIM(progress_results), '') IS NOT NULL THEN CONCAT('Progress results: ', TRIM(progress_results)) END,
            CASE WHEN NULLIF(TRIM(endreport_summary), '') IS NOT NULL THEN CONCAT('Final report summary: ', TRIM(endreport_summary)) END,
            CASE WHEN NULLIF(TRIM(endreport_results), '') IS NOT NULL THEN CONCAT('Final report results: ', TRIM(endreport_results)) END,
            CASE WHEN NULLIF(TRIM(keywords_json), '') IS NOT NULL THEN CONCAT('Keywords: ', TRIM(keywords_json)) END
        ) AS award_description,
        COALESCE(
            NULLIF(TRIM(related_subsidies_json), ''),
            NULLIF(TRIM(related_programs_json), ''),
            NULLIF(TRIM(main_subject), '')
        ) AS scheme_source
    FROM openalex.awards.zonmw_raw
)
SELECT
    abs(xxhash64(CONCAT('4320321007:zonmw:', LOWER(TRIM(rp.slug))))) % 9000000000 AS id,
    rp.display_name,
    NULLIF(rp.award_description, '') AS description,
    4320321007 AS funder_id,
    rp.slug AS funder_award_id,
    rp.amount_double AS amount,
    CASE WHEN rp.amount_double IS NOT NULL THEN 'EUR' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    rp.scheme_source AS funder_scheme,
    'zonmw_projects_jsonapi' AS provenance,
    rp.start_date_cast AS start_date,
    rp.end_date_cast AS end_date,
    rp.start_year_int AS start_year,
    rp.end_year_int AS end_year,
    CASE
        WHEN rp.leader_full_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                CAST(NULL AS STRING) AS given_name,
                rp.leader_full_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320321007:zonmw:', LOWER(TRIM(rp.slug))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
JOIN funder_resolved f ON f.funder_id = 4320321007
WHERE rp.project_number IS NOT NULL
  AND rp.display_name IS NOT NULL
  AND rp.start_year_int IS NOT NULL;


## Step 3: Insert into `openalex_awards_raw` at priority 92

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'zonmw_projects_jsonapi' AND priority = 92;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    92 AS priority
FROM openalex.awards.zonmw_awards;


## Step 6: Verification

Grant pattern with source-field amount waiver. The public API exposes project-budget columns, but current records do not populate them. Amount/currency should be non-null only if ZonMw starts publishing `field_project_budget_api`.


In [ ]:
%sql
SELECT COUNT(*) AS total_zonmw_awards
FROM openalex.awards.zonmw_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.zonmw_awards;


In [ ]:
%sql
-- Duplicate funder_award_id must be zero; duplicate ids must also be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.zonmw_awards;


In [ ]:
%sql
-- Data completeness / amount waiver check
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_pi_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_pi_affiliation,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description,
    ROUND(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_positive_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_start_date,
    collect_set(currency) AS currencies
FROM openalex.awards.zonmw_awards;


In [ ]:
%sql
SELECT
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    percentile_approx(amount, 0.5) AS median_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.zonmw_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS awards
FROM openalex.awards.zonmw_awards
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) AS awards
FROM openalex.awards.zonmw_awards
GROUP BY funding_type, funder_scheme
ORDER BY awards DESC
LIMIT 30;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 80) AS title,
    funder_award_id,
    start_date,
    end_date,
    lead_investigator.family_name AS project_leader,
    SUBSTRING(lead_investigator.affiliation.name, 1, 60) AS affiliation,
    landing_page_url
FROM openalex.awards.zonmw_awards
ORDER BY start_year DESC, id
LIMIT 20;


In [ ]:
%sql
-- Confirm the priority insert landed as expected.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'zonmw_projects_jsonapi'
GROUP BY priority, provenance;
